# Codigo genetico, entropia y biologia

Aplicamos teoria de la informacion al codigo genetico: calculamos entropia, redundancia e informacion de Fisher.

Articulo: `05/07-informacion-y-biologia.md`

In [ ]:
import math
import random
from collections import Counter

## 1. Entropia del alfabeto de bases

El ADN usa 4 bases: A, T, C, G. Si la distribucion fuera uniforme, la entropia maxima seria log2(4)=2 bits por base.
En el genoma humano real la distribucion no es exactamente uniforme.

In [ ]:
# Frecuencias aproximadas en el genoma humano (por ciento)
bases_humano = {'A': 0.295, 'T': 0.295, 'C': 0.205, 'G': 0.205}

def entropia(distribucion):
    return -sum(p * math.log2(p) for p in distribucion.values() if p > 0)

H_humano = entropia(bases_humano)
H_max = math.log2(4)
redundancia = 1 - H_humano / H_max

print("Distribucion de bases en el genoma humano:")
for b, p in bases_humano.items():
    print(f"  {b}: {p:.3f}  ({p*100:.1f}%)")
print(f"\nEntropia: H = {H_humano:.4f} bits/base")
print(f"Entropia maxima (uniforme): {H_max:.4f} bits/base")
print(f"Redundancia: {redundancia:.4f} ({redundancia*100:.1f}%)")

## 2. El codigo genetico como tabla de traduccion

El codigo genetico mapea 64 codones (tripletes) a 20 aminoacidos + STOP.
Esta degenerado: varios codones codifican el mismo aminoacido (redundancia).

In [ ]:
# Codigo genetico standard (64 codones -> 20 aa + STOP)
codigo_genetico = {
    'TTT':'Phe','TTC':'Phe','TTA':'Leu','TTG':'Leu',
    'CTT':'Leu','CTC':'Leu','CTA':'Leu','CTG':'Leu',
    'ATT':'Ile','ATC':'Ile','ATA':'Ile','ATG':'Met',
    'GTT':'Val','GTC':'Val','GTA':'Val','GTG':'Val',
    'TCT':'Ser','TCC':'Ser','TCA':'Ser','TCG':'Ser',
    'CCT':'Pro','CCC':'Pro','CCA':'Pro','CCG':'Pro',
    'ACT':'Thr','ACC':'Thr','ACA':'Thr','ACG':'Thr',
    'GCT':'Ala','GCC':'Ala','GCA':'Ala','GCG':'Ala',
    'TAT':'Tyr','TAC':'Tyr','TAA':'STOP','TAG':'STOP',
    'CAT':'His','CAC':'His','CAA':'Gln','CAG':'Gln',
    'AAT':'Asn','AAC':'Asn','AAA':'Lys','AAG':'Lys',
    'GAT':'Asp','GAC':'Asp','GAA':'Glu','GAG':'Glu',
    'TGT':'Cys','TGC':'Cys','TGA':'STOP','TGG':'Trp',
    'CGT':'Arg','CGC':'Arg','CGA':'Arg','CGG':'Arg',
    'AGT':'Ser','AGC':'Ser','AGA':'Arg','AGG':'Arg',
    'GGT':'Gly','GGC':'Gly','GGA':'Gly','GGG':'Gly',
}

# Degeneracion: cuantos codones por aminoacido
degeneracion = Counter(codigo_genetico.values())
print("Degeneracion del codigo genetico:")
for aa, n in sorted(degeneracion.items(), key=lambda x: -x[1]):
    print(f"  {aa:4s}: {n} codones  {'*'*n}")

print(f"\nTotal codones: {len(codigo_genetico)}")
print(f"Aminoacidos distintos (+ STOP): {len(degeneracion)}")
H_codones = math.log2(64)
H_aa = math.log2(21)
print(f"\nInformacion por codon (max): {H_codones:.2f} bits")
print(f"Informacion por aminoacido (max): {H_aa:.2f} bits")
print(f"Redundancia media del codigo: {(H_codones - H_aa)/H_codones:.2%}")

## 3. Robustez del codigo: mutaciones sinonimas

Una mutacion sinonima cambia un codon por otro que codifica el mismo aminoacido.
Esto es una forma de correccion de errores: las mutaciones en la tercera posicion (wobble) suelen ser sinonimas.

In [ ]:
def es_sinonima(codon1, codon2):
    return codigo_genetico.get(codon1) == codigo_genetico.get(codon2)

# Analizar cuantas mutaciones de 1 base son sinonimas
total_mutaciones = 0
mutaciones_sinonimas = 0
for codon, aa in codigo_genetico.items():
    if aa == 'STOP':
        continue
    for pos in range(3):
        for nueva_base in 'ACGT':
            if nueva_base != codon[pos]:
                nuevo_codon = codon[:pos] + nueva_base + codon[pos+1:]
                total_mutaciones += 1
                if es_sinonima(codon, nuevo_codon):
                    mutaciones_sinonimas += 1

print(f"Total de mutaciones de 1 base exploradas: {total_mutaciones}")
print(f"Mutaciones sinonimas: {mutaciones_sinonimas}")
print(f"Fraccion sinonima: {mutaciones_sinonimas/total_mutaciones:.3f}")
print()
print("Por posicion de la mutacion:")
for pos in range(3):
    total_pos, sin_pos = 0, 0
    for codon, aa in codigo_genetico.items():
        if aa == 'STOP': continue
        for nueva_base in 'ACGT':
            if nueva_base != codon[pos]:
                nuevo_codon = codon[:pos] + nueva_base + codon[pos+1:]
                total_pos += 1
                if es_sinonima(codon, nuevo_codon):
                    sin_pos += 1
    nombre = {0:'primera', 1:'segunda', 2:'tercera (wobble)'}[pos]
    print(f"  Posicion {nombre}: {sin_pos}/{total_pos} = {sin_pos/total_pos:.3f} sinonimas")

## 4. Informacion de Fisher en evolucion

La informacion de Fisher mide cuanta informacion sobre un parametro theta contiene una muestra.
En evolucion, cuantifica la capacidad de seleccion natural para discriminar entre fenotipos.

In [ ]:
def informacion_fisher_binomial(theta, n=100):
    """Informacion de Fisher de una muestra binomial para estimar theta."""
    # I(theta) = n / (theta * (1-theta)) para Bernoulli
    return n / (theta * (1 - theta))

thetas = [0.1, 0.3, 0.5, 0.7, 0.9]
print("Informacion de Fisher I(theta) para distribucion Bernoulli (n=100):")
print(f"{'theta':>8} {'I(theta)':>12} {'Var(estimador)':>16}")
for t in thetas:
    I = informacion_fisher_binomial(t)
    var = 1/I  # cota de Cramer-Rao
    print(f"{t:>8.1f} {I:>12.2f} {var:>16.4f}")

print()
print("Interpretacion: mayor I(theta) = seleccion mas eficiente para discriminar.")
print("En theta=0.5 la seleccion es menos eficiente (maxima varianza del estimador).")
print("En los extremos (fenotipos raros o comunes), la seleccion distingue mejor.")